# Berry–Esseen: auxiliary24: final join of the nine completed evidence bundles

Select **Runtime -> Run all**. No command or parameter edits are required.
Authorize Google Drive when requested. Keep one session per selected task.

Run this notebook LAST, after notebooks 14–22 have finished successfully and
their results ZIPs have been saved on the same Google Drive. Those preceding
jobs may run in parallel; this final join depends on all nine completed bundles.
It does not require restarting notebooks 01–07 or 09/10.

The join discovers the completed auxiliary21–23 and stronger24 B0–B5 evidence
on Drive, then checks its precise source and receipt identities, complete stage
censuses, captured outcomes and mathematical interfaces. A filename or historical
PASS label does not establish a fresh computation. Missing, partial or incompatible
evidence cannot become a successful join.

This revision also checks the actual preserved launch, supervisor result,
native worker/controller guard receipts and captures for every stage. It
keeps BASIC stages explicitly conditional and does not rerun the numerical
jobs. Existing valid14--22 exports can be used without restarting them.

This notebook assembles recorded evidence rather than rerunning the heavy numerical
jobs. The stated auxiliary fact is C24 < CE + 1/80, not the sharp C24 bound.
Remaining analytic premises, execution-contract limitations and the global proof
remain explicit. The final scoped result and an evidence ZIP are saved to Drive.


In [ ]:
from pathlib import Path
import subprocess, sys, json, os, time
CODE_COMMIT = "62593a0b0cf8fcf2c7b61666bf5e58cb9bb0a378"
TASK = "auxiliary24-final-join"
REPOSITORY = "https://github.com/rodziewiczk/berry-esseen-end2end.git"
CODE = Path("/content") / ("berry-esseen-code-" + CODE_COMMIT[:12])
print("Fetching the pinned source commit:", CODE_COMMIT, flush=True)
if not CODE.exists():
    subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPOSITORY, str(CODE)], check=True)
    subprocess.run(["git", "-C", str(CODE), "checkout", "--quiet", "--detach", CODE_COMMIT], check=True)
observed = subprocess.check_output(["git", "-C", str(CODE), "rev-parse", "HEAD"], text=True).strip()
if observed != CODE_COMMIT:
    raise RuntimeError("Source commit mismatch")
subprocess.run(["git", "-C", str(CODE), "diff", "--exit-code", "HEAD", "--"], check=True)
# A rerun in an existing Colab kernel must not reuse modules from an older pin.
for module_name in tuple(sys.modules):
    if module_name == "be_replay" or module_name.startswith("be_replay."):
        del sys.modules[module_name]
sys.path.insert(0, str(CODE))
from be_replay.environment import snapshot
if not Path(sys.modules["be_replay"].__file__).resolve().is_relative_to(CODE.resolve()):
    raise RuntimeError("Imported execution code does not belong to the pinned checkout")
print(json.dumps(snapshot(), indent=2), flush=True)
print("Source code and environment recorded. Mathematical acceptance has not yet been established.", flush=True)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DURABLE = Path("/content/drive/MyDrive/Berry-Esseen-end2end")
DURABLE.mkdir(parents=True, exist_ok=True)
print("Logs and checkpoints:", DURABLE, flush=True)


In [ ]:
from datetime import datetime, timezone
SELFTEST = DURABLE / ("selftest-" + TASK + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + ".log")
print("Testing the supervisor on the actual Linux runtime. These lightweight fixtures do not verify the mathematics.", flush=True)
with SELFTEST.open("w", encoding="utf-8") as log:
    child = subprocess.Popen([sys.executable, "-B", "-S", "-m", "unittest", "discover",
                              "-s", "tests", "-p", "test_linux_supervisor.py", "-v"],
                             cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in child.stdout:
        print(line, end="", flush=True)
        log.write(line); log.flush()
    code = child.wait()
if code != 0:
    raise RuntimeError("Supervisor self-tests failed; computation will not start. Log: " + str(SELFTEST))
print("Supervisor self-tests passed. Starting the selected calculation.", flush=True)


In [ ]:
print("Checking this job's orchestration and source-binding fixtures. No mathematics is executed by these tests.", flush=True)
DRIVER_SELFTEST = SELFTEST.with_name(SELFTEST.stem + "-driver.log")
with DRIVER_SELFTEST.open("w", encoding="utf-8") as log:
    result = subprocess.run([sys.executable, "-B", "-S", "-m", "unittest", "discover",
                             "-s", "tests", "-p", "test_aux24*.py", "-v"],
                            cwd=CODE, stdout=log, stderr=subprocess.STDOUT, text=True)
fixture_lines = DRIVER_SELFTEST.read_text(encoding="utf-8").splitlines()
if result.returncode == 0:
    summary = next((line for line in reversed(fixture_lines) if line.startswith("Ran ")), "All selected fixtures completed.")
    print("Driver infrastructure fixtures: PASS.", summary, flush=True)
else:
    print("\n".join(fixture_lines), flush=True)
print("Infrastructure fixture log:", DRIVER_SELFTEST, flush=True)
if result.returncode != 0:
    raise RuntimeError("The selected driver failed its infrastructure fixtures; computation will not start.")


In [ ]:
from be_replay.aux24_join import run_colab
RESULT = run_colab(durable_root=DURABLE)


In [ ]:
from IPython.display import display, FileLink
from google.colab import files
print("Execution status:", RESULT["status"]["state"], flush=True)
print("The global proof remains unverified.", flush=True)
print("Results ZIP on Google Drive:", RESULT["bundle"], flush=True)
display(FileLink(RESULT["bundle"]))
try:
    files.download(RESULT["bundle"])
except Exception as error:
    print("Download the ZIP from the Google Drive path shown above:", error, flush=True)


In [ ]:
# FINAL CELL: flush saved Drive results and release the Colab runtime.
# Runs after the final results ZIP is saved, including WAITING/INCOMPLETE.
# Do not run this cell manually while calculations or result writes are active.
# A synchronization error stops this cell BEFORE the runtime is released.
import os as _shutdown_os
import sys as _shutdown_sys

_managed_colab = (globals().get("BERRY_ESSEEN_DRIVE_MOUNTED") is True
                  or bool(_shutdown_os.environ.get("TBE_RUNTIME_ADDR")))
if not _managed_colab:
    print("Outside a managed Colab runtime; skipping shutdown.", flush=True)
else:
    from google.colab import drive as _shutdown_drive
    from google.colab import runtime as _shutdown_runtime

    print("Flushing saved results to Google Drive...", flush=True)
    _shutdown_sys.stdout.flush()
    _shutdown_sys.stderr.flush()
    _shutdown_os.sync()
    _shutdown_drive.flush_and_unmount(timeout_ms=300_000)
    print("Drive synchronization finished. Releasing the Colab runtime.", flush=True)
    _shutdown_runtime.unassign()
